# 📓 Notebook 06 — Vector Databases---**Phase 2 · Retrieval-Augmented Generation**> Scale your search to millions of vectors with persistent, production-grade vector databases.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| FAISS | Facebook's fast similarity search (local) || ChromaDB | Developer-friendly persistent vector store || ANN algorithms | HNSW, IVF — how approximate search works || Persistence | Save and reload your index || Comparison | When to use which database |### 🏗️ Build: Persistent Local Knowledge Base

## 1. Setup

In [ ]:
import os, json, timeimport numpy as npfrom dotenv import load_dotenvimport litellmload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")DEFAULT_EMBEDDING = os.getenv("DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small")def get_embedding(text, model=None):    model = model or DEFAULT_EMBEDDING    return litellm.embedding(model=model, input=[text]).data[0]["embedding"]def get_embeddings_batch(texts, model=None):    model = model or DEFAULT_EMBEDDING    return [d["embedding"] for d in litellm.embedding(model=model, input=texts).data]print(f"🤖 {DEFAULT_MODEL} | 📐 {DEFAULT_EMBEDDING}")

---## 2. Why Vector Databases?Our numpy approach (Notebook 03-05) has limits:| Problem | numpy Search | Vector DB ||---------|-------------|-----------|| Speed at scale | O(n) linear scan | O(log n) approximate || 1M vectors | ~seconds per query | ~milliseconds || Persistence | In-memory only | Disk-backed || Metadata | Manual implementation | Built-in filtering || Updates | Rebuild everything | Add/delete individual items |### ANN (Approximate Nearest Neighbor) Algorithms| Algorithm | How It Works | Trade-off ||-----------|-------------|-----------|| **HNSW** | Multi-layer graph, navigate to nearest | Best quality, more memory || **IVF** | Cluster vectors, search relevant clusters only | Good balance || **PQ** | Compress vectors with product quantization | Least memory, lower quality |> **Interview Tip:** *"How does HNSW work?"*  > It builds a multi-layer navigable small world graph. Top layers are sparse (for fast navigation), bottom layers are dense (for precision). Search starts at top and zooms in.

---## 3. FAISS — Facebook AI Similarity Search

In [ ]:
import faiss# Sample datadocuments = [    "Python is great for data science and machine learning",    "JavaScript powers the modern web with React and Node.js",    "Kubernetes orchestrates containers in production clusters",    "PostgreSQL is a powerful open-source relational database",    "Docker simplifies application deployment with containers",    "TensorFlow and PyTorch are leading deep learning frameworks",    "Redis provides in-memory caching for fast data access",    "GraphQL offers flexible API queries for frontend developers",    "Git enables collaborative version control for codebases",    "Prometheus monitors system metrics and alerts on anomalies",    "Apache Kafka handles high-throughput event streaming",    "Nginx serves as a reverse proxy and load balancer",    "MongoDB stores data in flexible JSON-like documents",    "Elasticsearch enables full-text search and log analytics",    "AWS Lambda runs serverless functions without managing servers",]# Embed all documentsprint("📐 Embedding documents...")embeddings = get_embeddings_batch(documents)emb_matrix = np.array(embeddings, dtype=np.float32)dimension = emb_matrix.shape[1]print(f"   Shape: {emb_matrix.shape} ({dimension} dimensions)")# ===== Index Type 1: Flat (Exact Search) =====index_flat = faiss.IndexFlatIP(dimension)  # Inner Product (= cosine for normalized vectors)faiss.normalize_L2(emb_matrix)             # Normalize for cosine similarityindex_flat.add(emb_matrix)print(f"\n✅ FAISS Flat Index: {index_flat.ntotal} vectors")# Searchquery = "container deployment"query_emb = np.array([get_embedding(query)], dtype=np.float32)faiss.normalize_L2(query_emb)distances, indices = index_flat.search(query_emb, k=3)print(f"\n🔍 Query: \"{query}\"")for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):    print(f"   {i+1}. [{dist:.4f}] {documents[idx]}")

In [ ]:
# ===== Index Type 2: IVF (Approximate, Faster) =====print("🏗️ Building IVF Index...")# IVF needs training on the datanlist = 4  # Number of clusters (use sqrt(n) as rule of thumb)quantizer = faiss.IndexFlatIP(dimension)index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist)# Train the indexindex_ivf.train(emb_matrix)index_ivf.add(emb_matrix)index_ivf.nprobe = 2  # Search 2 clusters (trade-off: more probes = slower but better recall)print(f"✅ IVF Index: {index_ivf.ntotal} vectors, {nlist} clusters")# Compare Flat vs IVFquery = "machine learning frameworks"query_emb = np.array([get_embedding(query)], dtype=np.float32)faiss.normalize_L2(query_emb)print(f"\n🔍 Query: \"{query}\"")# Flat (exact)start = time.time()d_flat, i_flat = index_flat.search(query_emb, k=3)flat_time = (time.time() - start) * 1000# IVF (approximate)start = time.time()d_ivf, i_ivf = index_ivf.search(query_emb, k=3)ivf_time = (time.time() - start) * 1000print(f"\n  Flat (exact):  {flat_time:.2f}ms — {[documents[i][:40] for i in i_flat[0]]}")print(f"  IVF (approx):  {ivf_time:.2f}ms — {[documents[i][:40] for i in i_ivf[0]]}")

In [ ]:
# Saving and loading FAISS indeximport tempfilesave_path = os.path.join(tempfile.gettempdir(), "faiss_index.bin")# Savefaiss.write_index(index_flat, save_path)print(f"💾 Saved index to: {save_path} ({os.path.getsize(save_path)} bytes)")# Loadloaded_index = faiss.read_index(save_path)print(f"📂 Loaded index: {loaded_index.ntotal} vectors")# Verify it worksd, i = loaded_index.search(query_emb, k=2)print(f"🔍 Search on loaded index: {[documents[idx][:50] for idx in i[0]]}")

---## 4. ChromaDB — Developer-Friendly Vector Store

In [ ]:
import chromadb# Create persistent ChromaDB clientpersist_dir = os.path.join(os.environ.get('TEMP', '/tmp'), "chroma_course")client = chromadb.PersistentClient(path=persist_dir)# Create or get a collectioncollection = client.get_or_create_collection(    name="tech_knowledge",    metadata={"hnsw:space": "cosine"},  # Use cosine similarity)# Add documents with metadataembeddings_list = get_embeddings_batch(documents)collection.add(    ids=[f"doc_{i}" for i in range(len(documents))],    documents=documents,    embeddings=embeddings_list,    metadatas=[        {"category": "python", "type": "language"},        {"category": "javascript", "type": "language"},        {"category": "kubernetes", "type": "devops"},        {"category": "postgresql", "type": "database"},        {"category": "docker", "type": "devops"},        {"category": "tensorflow", "type": "ml"},        {"category": "redis", "type": "database"},        {"category": "graphql", "type": "api"},        {"category": "git", "type": "tool"},        {"category": "prometheus", "type": "monitoring"},        {"category": "kafka", "type": "streaming"},        {"category": "nginx", "type": "devops"},        {"category": "mongodb", "type": "database"},        {"category": "elasticsearch", "type": "search"},        {"category": "aws", "type": "cloud"},    ])print(f"✅ ChromaDB collection: {collection.count()} documents")print(f"   Persistent path: {persist_dir}")

In [ ]:
# Search with ChromaDBquery = "database solutions"query_emb = get_embedding(query)results = collection.query(    query_embeddings=[query_emb],    n_results=3,)print(f"🔍 ChromaDB Query: \"{query}\"")for i, (doc, dist, meta) in enumerate(zip(results["documents"][0], results["distances"][0], results["metadatas"][0])):    print(f"   {i+1}. [{1-dist:.4f}] {doc}")    print(f"      Metadata: {meta}")# Search with metadata filterprint(f"\n🏷️ Filtered: type=database only")filtered = collection.query(    query_embeddings=[query_emb],    n_results=3,    where={"type": "database"},)for doc, dist in zip(filtered["documents"][0], filtered["distances"][0]):    print(f"   [{1-dist:.4f}] {doc}")

In [ ]:
# CRUD operations in ChromaDB# Update a documentcollection.update(    ids=["doc_0"],    documents=["Python 3.12 introduces significant performance improvements for data science and AI/ML workloads."],    embeddings=[get_embedding("Python 3.12 introduces significant performance improvements for data science and AI/ML workloads.")],)print("✅ Updated doc_0")# Delete a documentcollection.delete(ids=["doc_14"])print(f"✅ Deleted doc_14, collection now has {collection.count()} documents")# List all collectionscollections = client.list_collections()print(f"\n📋 Collections: {[c.name for c in collections]}")

---## 5. Vector Database Comparison| Feature | FAISS | ChromaDB | Pinecone ||---------|-------|----------|----------|| **Type** | Library | Embedded DB | Cloud service || **ANN Algorithm** | IVF, HNSW, PQ | HNSW | Proprietary || **Persistence** | Manual (save/load) | Built-in | Cloud-managed || **Metadata Filter** | No | Yes | Yes || **CRUD** | Add only | Full CRUD | Full CRUD || **Scale** | Millions (single machine) | Thousands-millions | Billions (distributed) || **Setup** | `pip install faiss-cpu` | `pip install chromadb` | API key + SDK || **Cost** | Free | Free | $$$ (cloud pricing) || **Best For** | High-performance local | Dev-friendly local | Production cloud |> **Interview Tip:** *"How would you choose a vector database?"*  > - **Prototyping / Small scale:** ChromaDB (easiest setup, full CRUD, metadata filtering)> - **Performance-critical / Large scale (single machine):** FAISS (fastest, most control)> - **Production cloud / Massive scale:** Pinecone or Weaviate (managed, distributed)

---## 📝 Interview Quiz — Vector Databases### Q1: Explain HNSW (Hierarchical Navigable Small World). Why is it popular?<details><summary>💡 Answer</summary>HNSW builds a multi-layer graph:- **Top layers**: Sparse graph with long-range connections (for fast initial navigation)- **Bottom layers**: Dense graph with short-range connections (for precise final search)Search: Start at top layer, greedily navigate to nearest node, descend layer by layer.**Why popular:**- Best recall/speed trade-off among ANN algorithms- No training step needed (unlike IVF)- Incrementally updatable (add/delete without rebuilding)- Used by ChromaDB, Pinecone, pgvector, Milvus</details>### Q2: What is the recall-speed trade-off in ANN search?<details><summary>💡 Answer</summary>**Exact search** (brute force): 100% recall, O(n) time**ANN search**: 95-99% recall, O(log n) timeThe trade-off is controlled by parameters:- HNSW: `ef_search` (higher = better recall, slower)- IVF: `nprobe` (more clusters searched = better recall, slower)**Rule of thumb:** Aim for 95%+ recall. Measure it by comparing ANN results against brute-force on a test set.</details>### Q3: How would you handle 100 million embeddings?<details><summary>💡 Answer</summary>1. **Compression:** Product Quantization (PQ) reduces vector size 4-8x2. **IVF + PQ:** Cluster first, compress within clusters3. **Sharding:** Split across multiple machines by ID range or metadata4. **GPU:** FAISS GPU can handle billions with hardware acceleration5. **Managed service:** Pinecone, Weaviate for auto-scaling6. **Disk-based:** FAISS OnDiskInvertedLists for larger-than-RAM indexes**Memory math:** 100M × 1536 dims × 4 bytes = 614 GB (raw). With PQ (8x): 77 GB.</details>### Q4: ChromaDB vs FAISS — when would you choose each?<details><summary>💡 Answer</summary>**Choose ChromaDB when:**- Need metadata filtering- Want easy document CRUD (update/delete)- Building RAG prototypes- Need persistence without manual save/load**Choose FAISS when:**- Maximum search speed is critical- Working with millions+ of vectors- Need GPU acceleration- Fine-grained control over index parameters- Memory-constrained environments (PQ compression)</details>

---## 🏗️ BUILD: Persistent Knowledge Base with RAG

In [ ]:
class PersistentKnowledgeBase:    """ChromaDB-backed knowledge base with full RAG."""        def __init__(self, collection_name="knowledge_base", persist_dir=None):        self.persist_dir = persist_dir or os.path.join(os.environ.get('TEMP', '/tmp'), "kb_data")        self.client = chromadb.PersistentClient(path=self.persist_dir)        self.collection = self.client.get_or_create_collection(            name=collection_name,            metadata={"hnsw:space": "cosine"},        )        self.llm_model = DEFAULT_MODEL        print(f"📚 Knowledge Base: {self.collection.count()} documents")        def add(self, texts, metadata_list=None, chunk_size=500):        """Add documents, auto-chunking if needed."""        chunks, metas, ids = [], [], []                for i, text in enumerate(texts):            # Simple chunking            for j in range(0, len(text), chunk_size):                chunk = text[j:j+chunk_size].strip()                if chunk:                    chunks.append(chunk)                    metas.append(metadata_list[i] if metadata_list else {"source": f"doc_{i}"})                    ids.append(f"chunk_{self.collection.count() + len(chunks) - 1}_{j}")                if chunks:            embeddings = get_embeddings_batch(chunks)            self.collection.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metas)            print(f"  ✅ Added {len(chunks)} chunks (total: {self.collection.count()})")        def search(self, query, top_k=3, where=None):        """Search the knowledge base."""        query_emb = get_embedding(query)        kwargs = {"query_embeddings": [query_emb], "n_results": top_k}        if where:            kwargs["where"] = where        return self.collection.query(**kwargs)        def ask(self, question, top_k=3, where=None):        """RAG: Search + Generate answer."""        results = self.search(question, top_k=top_k, where=where)                if not results["documents"][0]:            return "No relevant information found."                context = "\n\n".join(results["documents"][0])        response = litellm.completion(            model=self.llm_model,            messages=[                {"role": "system", "content": "Answer based on the provided context. Be precise and cite sources."},                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}            ],            temperature=0, max_tokens=500,        )        return response.choices[0].message.content        def stats(self):        print(f"  Total chunks: {self.collection.count()}")        print(f"  Persist dir: {self.persist_dir}")# Create and populate the knowledge basekb = PersistentKnowledgeBase("demo_kb")kb.add([    "Python was created by Guido van Rossum in 1991. It emphasizes code readability with significant indentation. Python supports multiple paradigms including procedural, OOP, and functional programming. It has a comprehensive standard library and a thriving ecosystem of third-party packages.",    "Rust is a systems programming language focused on safety, speed, and concurrency. It prevents common bugs through its ownership system which guarantees memory safety without a garbage collector. Rust has been voted the most loved language on Stack Overflow surveys.",    "Go (Golang) was designed at Google by Robert Griesemer, Rob Pike, and Ken Thompson. It features built-in concurrency with goroutines and channels. Go compiles to machine code and is used for cloud infrastructure, network services, and CLI tools.",], metadata_list=[    {"topic": "python", "year": 1991},    {"topic": "rust", "year": 2010},    {"topic": "go", "year": 2009},])# Query the persistent knowledge baseprint("\n" + "=" * 60)for q in ["Which language was created at Google?", "Tell me about memory safety", "Best language for beginners"]:    print(f"\n❓ {q}")    answer = kb.ask(q)    print(f"📝 {answer}")    print("-" * 40)

---## ✅ Notebook 06 Summary| Concept | Key Takeaway ||---------|-------------|| FAISS | Fast local search, IVF/HNSW/PQ index types || ChromaDB | Developer-friendly, persistent, metadata filtering || HNSW | Multi-layer graph, best recall/speed trade-off || IVF | Cluster-based search, needs training || Persistence | Save/load indexes for production use || Scale | FAISS for millions, Pinecone for billions |### ➡️ Next: [Notebook 07 — Tool Usage](./07_tools.ipynb)*Move from "chatbot" to "agent" — let LLMs use tools to take actions.*